# MedGemma CRC histology QLoRA — v36 plain Hugging Face fallback (OOM fix)

This notebook continues from v35. v35 proved the plain Hugging Face MedGemma QLoRA path is numerically valid: the one-batch active-digit preflight loss was finite. The next failure was CUDA OOM during training inside SigLIP attention at `pixel_values=(1, 3, 896, 896)`.

v36 fixes that by making the image encoder a frozen, no-gradient feature extractor during training. The project still uses MedGemma image inputs and trains LoRA adapters on the language-side layers, but it avoids backpropagating through the expensive 896x896 SigLIP vision tower.

Key changes:
- prevents dependency resolution from pulling `torch==2.11.0`;
- freezes all vision tower and multimodal projector parameters **after** LoRA injection;
- patches `get_image_features(...)` to run under `torch.no_grad()`;
- enables gradient checkpointing for the trainable language path;
- adds a one-step backward/memory preflight before `trainer.train()`.


In [ ]:
# Run this first, before importing torch / transformers.
# Plain Hugging Face QLoRA stack. No Unsloth imports.
# v36 fix: prevent pip from pulling torch==2.11.0 via transitive dependencies.
import os, sys, subprocess
from pathlib import Path
from importlib import metadata

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:64')

MARKER = Path('/kaggle/working/.medgemma_plain_hf_qlora_v36_installed')
OLD_MARKERS = [
    Path('/kaggle/working/.medgemma_plain_hf_qlora_v35_installed'),
]
PINNED = {
    'torch': '2.6.0',
    'torchvision': '0.21.0',
    'torchaudio': '2.6.0',
    'transformers': '4.56.2',
    'tokenizers': '0.22.1',
    'peft': '0.18.0',
    'accelerate': '1.10.1',
    'bitsandbytes': '0.46.0',
    'datasets': '4.3.0',
    'huggingface-hub': '0.36.0',
    'numpy': '1.26.4',
    'pandas': '2.2.2',
    'matplotlib': '3.8.4',
    'pillow': '11.3.0',
}

CHECK_PACKAGES = [
    'torch','torchvision','torchaudio','transformers','tokenizers','peft','accelerate',
    'bitsandbytes','datasets','huggingface-hub','numpy','pandas','matplotlib','pillow'
]

def pkg(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def print_versions():
    versions = {k: pkg(k) for k in CHECK_PACKAGES}
    for k, v in versions.items():
        print(f'{k}: {v}')
    return versions

def ok():
    versions = print_versions()
    required = {
        'torch': lambda v: v is not None and v.startswith(PINNED['torch']),
        'torchvision': lambda v: v is not None and v.startswith(PINNED['torchvision']),
        'torchaudio': lambda v: v is not None and v.startswith(PINNED['torchaudio']),
        'transformers': lambda v: v == PINNED['transformers'],
        'tokenizers': lambda v: v == PINNED['tokenizers'],
        'peft': lambda v: v == PINNED['peft'],
        'accelerate': lambda v: v == PINNED['accelerate'],
        'bitsandbytes': lambda v: v == PINNED['bitsandbytes'],
        'datasets': lambda v: v == PINNED['datasets'],
        'huggingface-hub': lambda v: v == PINNED['huggingface-hub'],
    }
    bad = [name for name, pred in required.items() if not pred(versions.get(name))]
    if bad:
        print('Dependency mismatch:', bad)
        return False
    # Import torch only after metadata looks correct.
    try:
        import torch
        print('CUDA available:', torch.cuda.is_available())
        if torch.cuda.is_available():
            print('GPU:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
            print('Torch CUDA:', torch.version.cuda)
        return True
    except Exception as exc:
        print('Torch import/version check failed:', repr(exc))
        return False

def run(cmd):
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd)

if MARKER.exists() and ok():
    print('Dependency check passed for v36. Continue to the next cell.')
else:
    print('Installing plain Hugging Face MedGemma QLoRA stack for v36.')
    print('Important fix: HF/PEFT packages are installed with --no-deps so pip cannot upgrade torch to 2.11.0.')
    loaded = [m for m in ('torch','torchvision','torchaudio','transformers','peft','bitsandbytes','datasets','huggingface_hub') if m in sys.modules]
    if loaded:
        print('Packages already imported in this kernel:', loaded)
        print('Installing then restarting so imports reload cleanly.')

    # Remove stale markers from previous notebooks so they do not short-circuit checks.
    for old in OLD_MARKERS:
        try:
            old.unlink()
        except FileNotFoundError:
            pass

    # Remove incompatible packages that caused earlier failures or unnecessary imports.
    run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])

    # Install the CUDA 12.4 PyTorch stack first.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
         'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
         '--index-url', 'https://download.pytorch.org/whl/cu124'])

    # Install application packages without dependency resolution, so Torch stays at 2.6.0.
    run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
         'transformers==4.56.2', 'tokenizers==0.22.1', 'peft==0.18.0',
         'accelerate==1.10.1', 'bitsandbytes==0.46.0', 'datasets==4.3.0',
         'huggingface-hub==0.36.0', 'hf_xet>=1.1.0,<2.0',
         'numpy==1.26.4', 'pandas==2.2.2', 'matplotlib==3.8.4',
         'pillow==11.3.0', 'tqdm', 'scikit-learn', 'safetensors', 'sentencepiece',
         'protobuf', 'packaging', 'regex', 'requests', 'pyyaml', 'filelock'])

    MARKER.write_text('ok')
    print('\nInstall complete. Restarting kernel now. After restart, run from the top again.')
    os._exit(0)




In [ ]:
import os, re, json, time, math, shutil, zipfile, random, urllib.request, gc
from pathlib import Path

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['TORCH_COMPILE_DISABLE'] = '1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:64')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from datasets import load_dataset
from huggingface_hub import login, hf_hub_download, whoami
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
    print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)

try:
    from importlib import metadata as importlib_metadata
    for pkg in ['transformers','tokenizers','peft','accelerate','bitsandbytes','datasets','huggingface-hub']:
        print(pkg + ':', importlib_metadata.version(pkg))
except Exception as exc:
    print('Version diagnostics unavailable:', repr(exc))

PROJECT_DIR = Path('/kaggle/working/MedGemma_PlainHF_QLoRA_CRC_v36')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
ADAPTER_DIR = PROJECT_DIR / 'medgemma_crc_plainhf_lora_v36'
for p in [PROJECT_DIR, DATA_DIR, RESULTS_DIR, ADAPTER_DIR]: p.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
LABEL_DESCRIPTIONS = {
    'ADI': 'adipose tissue / fat', 'BACK': 'empty background / no tissue',
    'DEB': 'debris / necrotic debris', 'LYM': 'lymphocytes / lymphoid cells',
    'MUC': 'mucus / mucin pools', 'MUS': 'smooth muscle',
    'NORM': 'normal colon mucosa', 'STR': 'cancer-associated stroma',
    'TUM': 'colorectal adenocarcinoma epithelium / tumor epithelium',
}
LABEL_TO_DIGIT = {label: str(i) for i, label in enumerate(LABEL_NAMES)}
DIGIT_TO_LABEL = {str(i): label for i, label in enumerate(LABEL_NAMES)}

CLASS_INSTRUCTION = """Classify this H&E colorectal histology image patch.
Return exactly one digit and nothing else.

0 = ADI, adipose tissue / fat
1 = BACK, empty background / no tissue
2 = DEB, debris / necrotic debris
3 = LYM, lymphocytes / lymphoid cells
4 = MUC, mucus / mucin pools
5 = MUS, smooth muscle
6 = NORM, normal colon mucosa
7 = STR, cancer-associated stroma
8 = TUM, colorectal adenocarcinoma epithelium / tumor epithelium

Answer with only one digit from 0 to 8."""

def simple_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred)) if len(y_true) else 0.0

def simple_confusion_matrix(y_true, y_pred, labels):
    pos = {l:i for i,l in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for t,p in zip(y_true, y_pred):
        if t in pos and p in pos: cm[pos[t], pos[p]] += 1
    return cm

def classification_report_df(y_true, y_pred, labels):
    cm = simple_confusion_matrix(y_true, y_pred, labels)
    rows=[]; total=cm.sum(); correct=np.trace(cm)
    for i,label in enumerate(labels):
        tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp; support=cm[i,:].sum()
        precision=tp/(tp+fp) if tp+fp else 0.0
        recall=tp/(tp+fn) if tp+fn else 0.0
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
        rows.append({'label':label,'precision':precision,'recall':recall,'f1-score':f1,'support':int(support)})
    acc=float(correct/total) if total else 0.0
    rows.append({'label':'accuracy','precision':acc,'recall':acc,'f1-score':acc,'support':int(total)})
    rows.append({'label':'macro avg','precision':float(np.mean([r['precision'] for r in rows[:len(labels)]])),'recall':float(np.mean([r['recall'] for r in rows[:len(labels)]])),'f1-score':float(np.mean([r['f1-score'] for r in rows[:len(labels)]])),'support':int(total)})
    weighted=sum(r['f1-score']*r['support'] for r in rows[:len(labels)]) / total if total else 0.0
    rows.append({'label':'weighted avg','precision':0.0,'recall':0.0,'f1-score':float(weighted),'support':int(total)})
    return pd.DataFrame(rows)

def plot_confusion_matrix(cm, labels, title, output_path):
    fig, ax = plt.subplots(figsize=(10,8))
    im = ax.imshow(cm, interpolation='nearest')
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), xticklabels=labels, yticklabels=labels, ylabel='True label', xlabel='Predicted label', title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): ax.text(j, i, str(cm[i,j]), ha='center', va='center')
    fig.tight_layout(); fig.savefig(output_path, dpi=160, bbox_inches='tight'); plt.show(); plt.close(fig)



In [ ]:
def get_hf_token():
    token = os.environ.get('HF_TOKEN', '').strip()
    if token: return token
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        if token: return token.strip()
    except Exception:
        pass
    print('HF_TOKEN not found. The official Google repo is gated; add Kaggle secret HF_TOKEN if the Google repo fails.')
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    try: print('Hugging Face account:', whoami(token=HF_TOKEN).get('name'))
    except Exception as exc: print('HF login worked, but whoami failed:', repr(exc))
else:
    print('No Hugging Face token in this runtime.')



In [ ]:
# Conservative Kaggle settings. Increase only after preflight and smoke evaluation work.
GOOGLE_MEDGEMMA_MODEL_ID = 'google/medgemma-1.5-4b-it'
UNSLOTH_BNB_MODEL_ID = 'unsloth/medgemma-1.5-4b-it-unsloth-bnb-4bit'
FALLBACK_UNSLOTH_BNB_MODEL_ID = 'unsloth/medgemma-1.5-4b-it-bnb-4bit'
MODEL_CANDIDATES = [GOOGLE_MEDGEMMA_MODEL_ID, UNSLOTH_BNB_MODEL_ID, FALLBACK_UNSLOTH_BNB_MODEL_ID]

TRAIN_PER_CLASS = 200
MAX_STEPS = 75
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-6
LORA_R = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.05
MAX_SEQUENCE_LENGTH = 768
SAVE_STEPS = 25
LOGGING_STEPS = 1
MAX_GRAD_NORM = 0.05
WARMUP_RATIO = 0.10
EVAL_MAX_SAMPLES = 500
EVAL_CHECKPOINT_EVERY = 50


# v36 OOM controls: keep images, but do not backpropagate through the 896x896 SigLIP vision tower.
FREEZE_VISION_AND_PROJECTOR_AFTER_LORA = True
PATCH_IMAGE_FEATURES_NO_GRAD = True
ENABLE_LANGUAGE_GRADIENT_CHECKPOINTING = True
RUN_BACKWARD_MEMORY_PREFLIGHT = True

# Critical stability knob: compute 4-bit matmuls in fp32 on T4.
BNB_4BIT_COMPUTE_DTYPE = torch.float32

print({'MODEL_CANDIDATES': MODEL_CANDIDATES, 'TRAIN_PER_CLASS': TRAIN_PER_CLASS, 'MAX_STEPS': MAX_STEPS, 'LORA_R': LORA_R, 'LEARNING_RATE': LEARNING_RATE, 'BNB_4BIT_COMPUTE_DTYPE': str(BNB_4BIT_COMPUTE_DTYPE), 'FREEZE_VISION_AND_PROJECTOR_AFTER_LORA': FREEZE_VISION_AND_PROJECTOR_AFTER_LORA, 'PATCH_IMAGE_FEATURES_NO_GRAD': PATCH_IMAGE_FEATURES_NO_GRAD})



In [ ]:
NCT_CRC_HE_100K_URL = 'https://zenodo.org/records/1214456/files/NCT-CRC-HE-100K.zip'
CRC_VAL_HE_7K_URL = 'https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip'

def looks_like_crc_folder(path: Path) -> bool:
    return path.is_dir() and all((path / label).is_dir() for label in LABEL_NAMES)

def find_crc_folder(folder_name: str):
    roots = [Path('/kaggle/input'), DATA_DIR, Path('/kaggle/working')]
    for root in roots:
        if not root.exists(): continue
        for candidate in root.glob(f'**/{folder_name}'):
            if looks_like_crc_folder(candidate): return candidate
    if folder_name == 'NCT-CRC-HE-100K':
        for candidate in Path('/kaggle/input').glob('*') if Path('/kaggle/input').exists() else []:
            if looks_like_crc_folder(candidate) and 'VAL' not in candidate.name.upper(): return candidate
    return None

def download_and_extract(url, zip_name, expected_folder):
    existing = find_crc_folder(expected_folder)
    if existing is not None:
        print(f'Found {expected_folder}:', existing); return existing
    zip_path = DATA_DIR / zip_name
    if not zip_path.exists():
        print('Downloading:', url); urllib.request.urlretrieve(url, zip_path); print('Downloaded:', zip_path)
    else:
        print('Using existing zip:', zip_path)
    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf: zf.extractall(DATA_DIR)
    extracted = find_crc_folder(expected_folder)
    if extracted is None: raise FileNotFoundError(f'Could not locate extracted {expected_folder}.')
    print(f'Found {expected_folder}:', extracted); return extracted

TRAIN_DIR = download_and_extract(NCT_CRC_HE_100K_URL, 'NCT-CRC-HE-100K.zip', 'NCT-CRC-HE-100K')
TEST_DIR = download_and_extract(CRC_VAL_HE_7K_URL, 'CRC-VAL-HE-7K.zip', 'CRC-VAL-HE-7K')
train_all = load_dataset('imagefolder', data_dir=str(TRAIN_DIR), split='train')
test_dataset = load_dataset('imagefolder', data_dir=str(TEST_DIR), split='train')
train_label_names = train_all.features['label'].names
test_label_names = test_dataset.features['label'].names
print('Train samples:', len(train_all), train_label_names)
print('Test samples:', len(test_dataset), test_label_names)



In [ ]:
def label_name_for_sample(sample, feature_label_names):
    return feature_label_names[int(sample['label'])]

def balanced_indices(dataset, feature_label_names, per_class, seed=SEED):
    rng = random.Random(seed)
    buckets = {label: [] for label in LABEL_NAMES}
    for idx, sample in enumerate(dataset):
        label = label_name_for_sample(sample, feature_label_names)
        if label in buckets: buckets[label].append(idx)
    chosen=[]
    for label in LABEL_NAMES:
        idxs=buckets[label]; rng.shuffle(idxs)
        take=len(idxs) if per_class is None else min(per_class, len(idxs))
        chosen.extend(idxs[:take]); print(f'{label}: selected {take} / {len(idxs)}')
    rng.shuffle(chosen); return chosen

print('Number of labels:', len(LABEL_NAMES), LABEL_NAMES)
train_indices = balanced_indices(train_all, train_label_names, TRAIN_PER_CLASS)
train_subset = train_all.select(train_indices)
print('Training subset size:', len(train_subset))
pd.DataFrame({'train_index': train_indices}).to_csv(RESULTS_DIR / 'selected_train_indices.csv', index=False)



In [ ]:
def convert_sample_to_conversation(sample, feature_label_names):
    label = label_name_for_sample(sample, feature_label_names)
    digit = LABEL_TO_DIGIT[label]
    image = sample['image']
    if hasattr(image, 'convert'): image = image.convert('RGB')
    return {
        'messages': [
            {'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': CLASS_INSTRUCTION}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': digit}]},
        ]
    }

class CRCConversationDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, feature_label_names):
        self.hf_dataset = hf_dataset; self.feature_label_names = feature_label_names
    def __len__(self): return len(self.hf_dataset)
    def __getitem__(self, idx): return convert_sample_to_conversation(self.hf_dataset[int(idx)], self.feature_label_names)

train_conversations = CRCConversationDataset(train_subset, train_label_names)
example = train_conversations[0]
print('Example prompt:', example['messages'][0]['content'][1]['text'][:220] + '...')
print('target digit:', example['messages'][1]['content'][0]['text'], '->', DIGIT_TO_LABEL[example['messages'][1]['content'][0]['text']])



In [ ]:
def can_read_config(repo_id, token=None):
    try:
        path = hf_hub_download(repo_id=repo_id, filename='config.json', token=token, force_download=False)
        print('Config OK:', repo_id, path)
        return True
    except Exception as exc:
        print('Config check failed:', repo_id, repr(exc))
        return False

selected_model_id = None
for candidate in MODEL_CANDIDATES:
    if can_read_config(candidate, HF_TOKEN):
        selected_model_id = candidate
        break
if selected_model_id is None:
    raise RuntimeError('Could not read config.json from any candidate. If using the Google repo, accept terms and add HF_TOKEN in Kaggle secrets.')

MEDGEMMA_MODEL_ID = selected_model_id
print('Selected model:', MEDGEMMA_MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
)

load_kwargs = dict(
    pretrained_model_name_or_path=MEDGEMMA_MODEL_ID,
    device_map='auto',
    trust_remote_code=True,
    token=HF_TOKEN,
    attn_implementation='eager',
    torch_dtype=torch.float16,
)
# The official Google repo is full precision and should be quantized locally.
# The Unsloth fallback repo may already carry a quantization_config; if so, try without overriding first.
if not MEDGEMMA_MODEL_ID.startswith('unsloth/'):
    load_kwargs['quantization_config'] = bnb_config

processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
try:
    model = AutoModelForImageTextToText.from_pretrained(**load_kwargs)
except Exception as exc:
    if MEDGEMMA_MODEL_ID.startswith('unsloth/'):
        print('Pre-quantized Unsloth repo load failed once; retrying with explicit local BitsAndBytesConfig.')
        print(repr(exc))
        load_kwargs['quantization_config'] = bnb_config
        model = AutoModelForImageTextToText.from_pretrained(**load_kwargs)
    else:
        raise

model.config.use_cache = False
if hasattr(model, 'generation_config'): model.generation_config.use_cache = False
print('Loaded plain HF model:', type(model))
print('is_loaded_in_4bit:', getattr(model, 'is_loaded_in_4bit', None))

# Keep the image encoder frozen for stability; train language-side LoRA only.
def set_module_requires_grad_by_name_ending(model, endings, requires_grad=False):
    hits=[]
    for name, module in model.named_modules():
        if any(name.endswith(e) for e in endings):
            for p in module.parameters(): p.requires_grad_(requires_grad)
            hits.append(name)
    return hits

frozen = set_module_requires_grad_by_name_ending(model, ['vision_tower', 'multi_modal_projector'], False)
print('Frozen vision/projector modules:', frozen[:6], 'count=', len(frozen))

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)


def freeze_parameters_containing(model, substrings):
    changed = []
    for name, param in model.named_parameters():
        if any(s in name for s in substrings):
            if param.requires_grad:
                changed.append(name)
            param.requires_grad_(False)
    return changed

def trainable_parameter_summary(model, max_rows=30):
    rows = []
    total = trainable = 0
    for name, param in model.named_parameters():
        n = param.numel(); total += n
        if param.requires_grad:
            trainable += n
            rows.append((name, n, str(param.dtype)))
    print(f'Trainable params after final freezing: {trainable:,} / {total:,} = {100*trainable/total:.4f}%')
    print('First trainable parameter names:')
    for row in rows[:max_rows]:
        print(' ', row)
    if len(rows) > max_rows:
        print(f'  ... {len(rows)-max_rows} more trainable tensors')
    return rows

def disable_vision_gradient_checkpointing(model):
    disabled = []
    for name, module in model.named_modules():
        if 'vision_tower' in name:
            if hasattr(module, 'gradient_checkpointing_disable'):
                try:
                    module.gradient_checkpointing_disable()
                    disabled.append(name + '.gradient_checkpointing_disable()')
                except Exception:
                    pass
            if hasattr(module, 'gradient_checkpointing'):
                try:
                    module.gradient_checkpointing = False
                    disabled.append(name + '.gradient_checkpointing=False')
                except Exception:
                    pass
    print('Vision gradient checkpointing disabled hooks/modules:', len(disabled))
    if disabled:
        print(' first:', disabled[0])
        print(' last :', disabled[-1])
    return disabled

def patch_get_image_features_no_grad(model):
    # Since the vision tower and projector are frozen, image features can be computed without autograd.
    # This keeps image inputs in the MedGemma forward pass while avoiding the training-time SigLIP attention OOM.
    import types
    patched = []
    for name, module in model.named_modules():
        if hasattr(module, 'get_image_features') and callable(getattr(module, 'get_image_features')):
            if getattr(module, '_crc_no_grad_get_image_features_patch', False):
                continue
            old_get_image_features = module.get_image_features
            def new_get_image_features(self, pixel_values, _old=old_get_image_features):
                with torch.no_grad():
                    return _old(pixel_values)
            module.get_image_features = types.MethodType(new_get_image_features, module)
            module._crc_no_grad_get_image_features_patch = True
            patched.append(name)
    print('Patched get_image_features with torch.no_grad on modules:', patched)
    return patched
# Re-freeze after prepare_model_for_kbit_training.
frozen = set_module_requires_grad_by_name_ending(model, ['vision_tower', 'multi_modal_projector'], False)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)

if FREEZE_VISION_AND_PROJECTOR_AFTER_LORA:
    frozen_after_lora = freeze_parameters_containing(model, ['vision_tower', 'multi_modal_projector'])
    print('Froze trainable vision/projector parameters after LoRA injection:', len(frozen_after_lora))
    for name in frozen_after_lora[:10]: print('  froze:', name)

if ENABLE_LANGUAGE_GRADIENT_CHECKPOINTING:
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        print('Enabled gradient checkpointing on model for trainable language path.')
    except TypeError:
        model.gradient_checkpointing_enable()
        print('Enabled gradient checkpointing on model for trainable language path.')
    except Exception as exc:
        print('Could not enable gradient checkpointing:', repr(exc))

disable_vision_gradient_checkpointing(model)
if PATCH_IMAGE_FEATURES_NO_GRAD:
    patch_get_image_features_no_grad(model)

model.print_trainable_parameters()
trainable_parameter_summary(model)

if torch.cuda.is_available():
    print('GPU allocated GB after load:', torch.cuda.memory_allocated() / 1024**3)
    print('GPU reserved GB after load:', torch.cuda.memory_reserved() / 1024**3)



In [ ]:
def extract_assistant_digit(example):
    for message in reversed(example.get('messages', [])):
        if message.get('role') != 'assistant': continue
        content = message.get('content', '')
        pieces=[]
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and item.get('type') == 'text': pieces.append(str(item.get('text','')))
                elif isinstance(item, str): pieces.append(item)
        else: pieces.append(str(content))
        text=''.join(pieces).strip()
        m=re.search(r'[0-8]', text)
        if m: return m.group(0)
    raise RuntimeError(f'Could not find assistant digit in example: {example}')

def find_last_subsequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence): return -1
    for start in range(len(sequence)-len(subsequence), -1, -1):
        if sequence[start:start+len(subsequence)] == subsequence: return start
    return -1

def candidate_digit_token_sequences(tokenizer, digit):
    candidates=[]
    for text in [digit, '\n'+digit, ' '+digit, digit+'<end_of_turn>']:
        ids=tokenizer(text, add_special_tokens=False).input_ids
        if ids and ids not in candidates: candidates.append(ids)
    return candidates

class PlainMedGemmaDigitCollator:
    def __init__(self, processor, max_length=768, pixel_dtype=torch.float32):
        self.processor=processor; self.tokenizer=processor.tokenizer; self.max_length=max_length; self.pixel_dtype=pixel_dtype
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    def __call__(self, examples):
        encs=[]; debug=[]
        for ex in examples:
            enc = self.processor.apply_chat_template(
                ex['messages'], add_generation_prompt=False, tokenize=True,
                return_dict=True, return_tensors='pt', padding=False,
            )
            row = {k: v.squeeze(0) for k,v in enc.items() if torch.is_tensor(v)}
            digit = extract_assistant_digit(ex)
            seq = row['input_ids'].detach().cpu().tolist()
            matches=[]
            for ids in candidate_digit_token_sequences(self.tokenizer, digit):
                start = find_last_subsequence(seq, ids)
                if start >= 0: matches.append((start, len(ids), ids))
            if not matches:
                tail=self.tokenizer.decode(seq[-80:], skip_special_tokens=False)
                raise RuntimeError(f'Could not locate assistant digit {digit!r}. Tail={tail!r}')
            start, span_len, ids = max(matches, key=lambda x: (x[0], -x[1]))
            labels = torch.full_like(row['input_ids'], -100)
            labels[start:start+span_len] = row['input_ids'][start:start+span_len]
            row['labels'] = labels
            debug.append({'digit': digit, 'start': int(start), 'span_len': int(span_len), 'ids': ids, 'decoded': self.tokenizer.decode(ids, skip_special_tokens=False)})
            encs.append(row)
        max_len = min(max(e['input_ids'].shape[0] for e in encs), self.max_length)
        batch={}
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0
        for key in ['input_ids', 'attention_mask', 'token_type_ids', 'labels']:
            if key not in encs[0]: continue
            fill = -100 if key == 'labels' else (0 if key != 'input_ids' else pad_id)
            rows=[]
            for e in encs:
                x=e[key][:max_len]
                if x.shape[0] < max_len:
                    pad = torch.full((max_len-x.shape[0],), fill, dtype=x.dtype)
                    x = torch.cat([x, pad], dim=0)
                rows.append(x)
            batch[key] = torch.stack(rows, dim=0)
        if 'pixel_values' in encs[0]:
            batch['pixel_values'] = torch.stack([e['pixel_values'].to(self.pixel_dtype) for e in encs], dim=0)
        batch['_digit_label_debug'] = debug
        return batch

class DigitOnlyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        inputs.pop('_digit_label_debug', None)
        outputs = model(**inputs, use_cache=False, return_dict=True)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :]
        shift_labels = labels[..., 1:]
        active = shift_labels != -100
        active_count = int(active.sum().detach().cpu().item())
        if active_count == 0:
            raise RuntimeError('Zero active digit labels after shift.')
        active_logits = shift_logits[active].float()
        active_labels = shift_labels[active].to(active_logits.device)
        finite = torch.isfinite(active_logits)
        if not bool(finite.all().detach().cpu().item()):
            bad = int((~finite).sum().detach().cpu().item()); total = int(active_logits.numel())
            raise RuntimeError(f'Non-finite logits at supervised digit positions: {bad}/{total}. Plain HF fallback also failed; switch to MedSigLIP/classifier or A100 bf16 official notebook.')
        loss = F.cross_entropy(active_logits, active_labels, reduction='mean')
        return (loss, outputs) if return_outputs else loss

collator = PlainMedGemmaDigitCollator(processor, max_length=MAX_SEQUENCE_LENGTH, pixel_dtype=torch.float32)
preview = collator([train_conversations[0]])
print('Preview batch dtypes:')
for k,v in preview.items():
    if k.startswith('_'): continue
    print(k, tuple(v.shape), v.dtype)
print('Digit debug:', preview['_digit_label_debug'])
sup = preview['labels'][0][preview['labels'][0] != -100]
print('Supervised label tokens:', int(sup.numel()), 'decoded:', repr(processor.tokenizer.decode(sup.tolist(), skip_special_tokens=False)))



In [ ]:
TRAINER_OUTPUT_DIR = PROJECT_DIR / 'trainer_outputs'
TRAINER_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

args = TrainingArguments(
    output_dir=str(TRAINER_OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_strategy='steps',
    save_total_limit=2,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,
    optim='adamw_torch',
    lr_scheduler_type='linear',
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    seed=SEED,
)

trainer = DigitOnlyTrainer(model=model, args=args, train_dataset=train_conversations, data_collator=collator, processing_class=processor)

print('Running plain-HF one-batch finite-loss preflight...')
preflight = collator([train_conversations[0]])
debug = preflight.pop('_digit_label_debug', None)
print('Preflight digit debug:', debug)
preflight_inputs = trainer._prepare_inputs(preflight)
model.train()
with torch.no_grad():
    loss = trainer.compute_loss(model, dict(preflight_inputs))
print('One-batch active-digit preflight loss:', float(loss.detach().cpu()))
if not torch.isfinite(loss.detach()).all():
    raise RuntimeError('Initial loss is non-finite in plain HF fallback.')
del preflight, preflight_inputs, loss
if torch.cuda.is_available(): torch.cuda.empty_cache()

if RUN_BACKWARD_MEMORY_PREFLIGHT:
    print('Running one-batch backward/memory preflight...')
    model.train()
    preflight = collator([train_conversations[0]])
    preflight.pop('_digit_label_debug', None)
    preflight_inputs = trainer._prepare_inputs(preflight)
    try:
        loss = trainer.compute_loss(model, dict(preflight_inputs))
        print('Backward preflight loss:', float(loss.detach().cpu()))
        loss.backward()
        print('Backward preflight succeeded.')
        model.zero_grad(set_to_none=True)
        if torch.cuda.is_available():
            print('GPU allocated GB after backward preflight:', round(torch.cuda.memory_allocated()/1024**3, 3))
            print('GPU reserved  GB after backward preflight:', round(torch.cuda.memory_reserved()/1024**3, 3))
    except torch.cuda.OutOfMemoryError:
        print('Backward preflight OOM. Try lowering MAX_SEQUENCE_LENGTH to 512 or TRAIN_PER_CLASS to 100.')
        raise
    finally:
        del preflight, preflight_inputs
        if 'loss' in locals(): del loss
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

class StopOnNonFiniteLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            try: x=float(logs['loss'])
            except Exception: return control
            if not math.isfinite(x):
                print('Non-finite loss detected; stopping.'); control.should_training_stop=True; control.should_epoch_stop=True
        return control
trainer.add_callback(StopOnNonFiniteLossCallback())

start=time.time()
train_result = trainer.train()
train_minutes=(time.time()-start)/60
print(f'Training finished in {train_minutes:.2f} minutes')
print(train_result)
with open(RESULTS_DIR / 'train_result.json', 'w') as f:
    json.dump({'train_minutes':train_minutes,'train_result':str(train_result),'model_id':MEDGEMMA_MODEL_ID}, f, indent=2)



In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
zip_path = shutil.make_archive(str(PROJECT_DIR / 'medgemma_crc_plainhf_lora_v36_adapter'), 'zip', ADAPTER_DIR)
print('Saved adapter:', ADAPTER_DIR)
print('Adapter zip:', zip_path)



In [ ]:
model.eval()

def move_inputs_to_device(inputs):
    moved={}
    for k,v in inputs.items():
        if torch.is_tensor(v):
            # Keep pixels float32 for stability; other floating tensors can stay as produced.
            if k == 'pixel_values' and torch.is_floating_point(v): v = v.to(torch.float32)
            moved[k] = v.to(model.device if hasattr(model, 'device') else DEVICE)
        else:
            moved[k]=v
    return moved

def build_prompt_inputs(image):
    if hasattr(image, 'convert'): image=image.convert('RGB')
    messages=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':CLASS_INSTRUCTION}]}]
    inputs=processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    return move_inputs_to_device(inputs)

def get_digit_token_ids(tokenizer):
    mapping={}
    for digit in [str(i) for i in range(9)]:
        variants=[]
        for v in [digit, ' '+digit, '\n'+digit]:
            toks=tokenizer(v, add_special_tokens=False).input_ids
            if len(toks)==1: variants.append((v,toks[0]))
        if not variants: raise RuntimeError(f'No single-token variant for digit {digit}')
        mapping[digit]=variants
    return mapping
DIGIT_TOKEN_IDS = get_digit_token_ids(processor.tokenizer)
print('Digit token IDs:', DIGIT_TOKEN_IDS)

def predict_digit_for_image(image):
    inputs=build_prompt_inputs(image)
    with torch.inference_mode():
        outputs=model(**inputs, use_cache=False, return_dict=True)
        logits=outputs.logits[:, -1, :].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError('Non-finite logits during prediction.')
        log_probs=torch.log_softmax(logits, dim=-1)[0]
    digit_scores={}
    for digit, variants in DIGIT_TOKEN_IDS.items():
        digit_scores[digit]=max(float(log_probs[token_id].item()) for _,token_id in variants)
    best=max(digit_scores, key=digit_scores.get)
    top=sorted(digit_scores.items(), key=lambda kv:kv[1], reverse=True)
    raw='forced_digit_top9=' + ', '.join(f'{DIGIT_TO_LABEL[d]}:{s:.3f}' for d,s in top)
    return DIGIT_TO_LABEL[best], raw, digit_scores



In [ ]:
smoke_n=min(24, len(test_dataset))
rows=[]
print(f'Running smoke test on {smoke_n} samples...')
for i in range(smoke_n):
    sample=test_dataset[i]
    true_label=label_name_for_sample(sample, test_label_names)
    pred_label, raw, scores=predict_digit_for_image(sample['image'])
    rows.append({'sample_index':i,'true_label':true_label,'predicted_label':pred_label,'raw_scores':raw})
    print(f'{i:02d} true={true_label:>5s} pred={pred_label:>5s} {raw[:100]}')
smoke_df=pd.DataFrame(rows)
smoke_df.to_csv(RESULTS_DIR / 'smoke_test_predictions.csv', index=False)
print('Smoke accuracy:', simple_accuracy(smoke_df.true_label, smoke_df.predicted_label))
print('Predicted classes:', sorted(smoke_df.predicted_label.unique().tolist()))



In [ ]:
eval_dataset = test_dataset
if EVAL_MAX_SAMPLES is not None:
    eval_dataset = eval_dataset.select(range(min(EVAL_MAX_SAMPLES, len(eval_dataset))))
TOTAL_EVAL=len(eval_dataset)
print('Evaluation samples:', TOTAL_EVAL)
PREDICTIONS_PATH=RESULTS_DIR / 'medgemma_plainhf_qlora_predictions.csv'
CHECKPOINT_PATH=RESULTS_DIR / 'medgemma_plainhf_qlora_predictions_checkpoint.csv'
rows=[]; processed=set()
if CHECKPOINT_PATH.exists():
    old=pd.read_csv(CHECKPOINT_PATH); rows=old.to_dict('records'); processed=set(old['sample_index'].astype(int).tolist())
    print('Loaded checkpoint rows:', len(rows))
start=time.time(); last=time.time()
for i in range(TOTAL_EVAL):
    if i in processed: continue
    if i % 25 == 0: print(f'Processing sample {i}/{TOTAL_EVAL}')
    sample=eval_dataset[i]
    true_label=label_name_for_sample(sample, test_label_names)
    pred_label, raw, scores=predict_digit_for_image(sample['image'])
    row={'sample_index':i,'true_label':true_label,'predicted_label':pred_label,'raw_scores':raw}
    for digit, score in scores.items(): row[f'score_{DIGIT_TO_LABEL[digit]}']=score
    rows.append(row)
    if (i+1) % EVAL_CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, index=False)
        now=time.time(); spm=EVAL_CHECKPOINT_EVERY / ((now-last)/60); last=now
        print(f'Saved checkpoint {i+1}/{TOTAL_EVAL}; speed {spm:.2f} samples/min')
    if torch.cuda.is_available() and (i+1)%25==0: torch.cuda.empty_cache()
results_df=pd.DataFrame(rows).sort_values('sample_index')
results_df.to_csv(PREDICTIONS_PATH, index=False)
results_df.to_csv(CHECKPOINT_PATH, index=False)
print('Saved predictions:', PREDICTIONS_PATH)
print('Evaluation minutes:', (time.time()-start)/60)
results_df.head()



In [ ]:
results_df=pd.read_csv(RESULTS_DIR / 'medgemma_plainhf_qlora_predictions.csv')
y_true=results_df['true_label'].tolist(); y_pred=results_df['predicted_label'].tolist()
accuracy=simple_accuracy(y_true, y_pred)
report=classification_report_df(y_true, y_pred, LABEL_NAMES)
cm=simple_confusion_matrix(y_true, y_pred, LABEL_NAMES)
metrics={'model':'medgemma_plainhf_qlora_v36','base_model':MEDGEMMA_MODEL_ID,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu','num_eval_samples':int(len(results_df)),'accuracy':accuracy,'macro_f1':float(report.loc[report['label']=='macro avg','f1-score'].iloc[0]),'weighted_f1':float(report.loc[report['label']=='weighted avg','f1-score'].iloc[0]),'train_per_class':TRAIN_PER_CLASS,'max_steps':MAX_STEPS,'lora_r':LORA_R,'learning_rate':LEARNING_RATE}
with open(RESULTS_DIR / 'medgemma_plainhf_qlora_metrics.json','w') as f: json.dump(metrics, f, indent=2)
report.to_csv(RESULTS_DIR / 'medgemma_plainhf_qlora_classification_report.csv', index=False)
pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES).to_csv(RESULTS_DIR / 'medgemma_plainhf_qlora_confusion_matrix.csv')
plot_confusion_matrix(cm, LABEL_NAMES, 'MedGemma Plain HF QLoRA Confusion Matrix', RESULTS_DIR / 'medgemma_plainhf_qlora_confusion_matrix.png')
print(json.dumps(metrics, indent=2)); print(report)
zip_path=shutil.make_archive(str(PROJECT_DIR / 'medgemma_plainhf_qlora_results'), 'zip', RESULTS_DIR)
print('Created results zip:', zip_path)
os.chdir('/kaggle/working')
try:
    from IPython.display import FileLink, display
    display(FileLink('MedGemma_PlainHF_QLoRA_CRC_v36/medgemma_plainhf_qlora_results.zip'))
except Exception as exc:
    print('FileLink unavailable:', exc)
    print('Download manually from:', zip_path)



In [ ]:
print('Download before ending Kaggle session:')
print('/kaggle/working/MedGemma_PlainHF_QLoRA_CRC_v36/medgemma_plainhf_qlora_results.zip')
print('/kaggle/working/MedGemma_PlainHF_QLoRA_CRC_v36/medgemma_crc_plainhf_lora_v36_adapter.zip')

